# Load casualty data: Gaza and Ukraine

Loads the two casualty datasets used in Problem 3 of Assignment 2 (Math 50)
into two dataframes, `gaza` and `ukraine`, each with columns `time`,
`counts` (cumulative reported count) and `daily` (day-over-day increment),
plus whatever other information is available in the source files. The
first 300 days of each series are dropped (low, volatile early counts).

`gaza` additionally has a `period` column (`"before"` / `"after"`) marking
each row relative to the October 10, 2025 ceasefire, instead of truncating
the series at that date.

Data is loaded directly from [`math50-data`](https://github.com/elevien/math50-data),
so this notebook runs as-is in Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
%config InlineBackend.figure_format = 'svg'

## `kernel_smooth`

A kernel smoother for an equally-spaced time series: it estimates the
underlying trend by averaging nearby points, weighted by how close they are
to each target time. The bandwidth `L` controls how wide that averaging
window is -- you don't need to worry about the details to use it below.

In [ ]:
def gaussian_kernel(u):
    return np.exp(-0.5 * u ** 2)


def kernel_smooth(y, L, kernel=gaussian_kernel):
    """Kernel-smooth an equally-spaced time series y, with bandwidth L."""
    y = np.asarray(y, dtype=float)
    t = np.arange(len(y), dtype=float)
    weights = kernel((t[:, None] - t[None, :]) / L)
    return weights @ y / weights.sum(axis=1)

## Load the data

In [ ]:
GAZA_URL = "https://raw.githubusercontent.com/elevien/math50-data/main/data/gaza-casualties.csv"
UKRAINE_URL = "https://raw.githubusercontent.com/elevien/math50-data/main/data/ukraine-personnel-losses.csv"

BURN = 300
GAZA_CEASEFIRE = pd.Timestamp("2025-10-10")


def load_gaza():
    df = pd.read_csv(GAZA_URL, parse_dates=["report_date"])
    counts = df["ext_killed_cum"].to_numpy().astype(float)
    daily = np.empty(len(counts))
    daily[0] = np.nan
    daily[1:] = counts[1:] - counts[:-1]

    period = np.empty(len(df), dtype=object)
    before = (df["report_date"] <= GAZA_CEASEFIRE).to_numpy()
    period[before] = "before"
    period[~before] = "after"

    out = pd.DataFrame({
        "time": df["report_date"],
        "counts": counts,
        "daily": daily,
        "period": period,
        "children_killed_cum": df["ext_killed_children_cum"].to_numpy(),
        "women_killed_cum": df["ext_killed_women_cum"].to_numpy(),
        "injured_cum": df["ext_injured_cum"].to_numpy(),
    })
    return out.iloc[BURN + 1:].reset_index(drop=True)


def load_ukraine():
    df = pd.read_csv(UKRAINE_URL, parse_dates=["date"])
    counts = df["personnel"].to_numpy().astype(float)
    daily = np.empty(len(counts))
    daily[0] = np.nan
    daily[1:] = counts[1:] - counts[:-1]

    out = pd.DataFrame({
        "time": df["date"],
        "counts": counts,
        "daily": daily,
        "POW": df["POW"].to_numpy(),
    })
    return out.iloc[BURN + 1:].reset_index(drop=True)


gaza = load_gaza()
ukraine = load_ukraine()
gaza

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(gaza.time, gaza.daily, ".", markersize=3)
ax.set_xlabel("Date")
ax.set_ylabel("Daily reported deaths")
plt.show()

In [ ]:
ukraine

## Using `kernel_smooth`

Example: smoothing the Gaza daily series with a bandwidth of `L = 14` days.

In [ ]:
smoothed = kernel_smooth(gaza.daily.to_numpy(), L=14)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(gaza.time, gaza.daily, ".", markersize=3, alpha=0.4, label="daily")
ax.plot(gaza.time, smoothed, label="kernel_smooth(L=14)")
ax.set_xlabel("Date")
ax.legend()
plt.show()